In [1]:
!pip install stable_baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 60.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [2]:
# DDPG Portfolio Recommender with Future Sell Signals

# First, mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from stable_baselines3 import DDPG
import os
import random

# Set paths to your model and data
model_path = "/content/drive/MyDrive/ddpg_portfolio_model.zip"
data_path = "/content/drive/MyDrive/historical_data.csv"

class DDPGPortfolioRecommender:
    """
    A portfolio recommender using DDPG model with future sell signal simulation
    """
    def __init__(self, model_path, data_path, max_stocks=100, lookback=30, feature_dimension=7):
        print(f"Loading model from: {model_path}")
        self.model = DDPG.load(model_path)
        self.max_stocks = max_stocks
        self.lookback = lookback
        self.feature_dimension = feature_dimension

        print(f"Loading data from: {data_path}")
        # Load the most recent data for prediction
        data = pd.read_csv(data_path)
        data['date'] = pd.to_datetime(data['date'])

        # Get the most recent date
        self.latest_date = data['date'].max()
        print(f"Latest data date: {self.latest_date}")

        # Select top stocks by volume
        recent_data = data[data['date'] >= (self.latest_date - pd.DateOffset(days=30))]
        avg_volumes = recent_data.groupby('tic')['volume'].mean()
        top_tickers = avg_volumes.nlargest(max_stocks).index.tolist()

        # Get the list of tickers
        self.tickers = sorted(top_tickers)
        print(f"Number of tickers selected: {len(self.tickers)}")

        # Store the latest prices
        latest_data = data[data['date'] == self.latest_date]
        self.latest_prices = {row['tic']: row['close'] for _, row in latest_data.iterrows()
                              if row['tic'] in self.tickers}

        # Track previous recommendations and purchase history
        self.previous_allocations = {}
        self.purchase_history = {}

        # Add technical indicators for prediction
        print("Preparing features...")
        self._prepare_features(data)
        print("Initialization complete!")

    def _prepare_features(self, data):
        """Prepare features for the model prediction"""
        # Filter for only the tickers we're using
        filtered_data = data[data['tic'].isin(self.tickers)]

        # Get the most recent dates for feature calculation
        recent_dates = sorted(filtered_data['date'].unique())[-self.lookback:]
        recent_data = filtered_data[filtered_data['date'].isin(recent_dates)]

        # Calculate features (matching the training features)
        features = np.zeros((self.max_stocks, self.lookback, self.feature_dimension), dtype=np.float32)

        print(f"Calculating features for {len(self.tickers)} stocks with {self.feature_dimension} features...")
        for i, ticker in enumerate(self.tickers):
            if i >= self.max_stocks:
                break

            ticker_data = recent_data[recent_data['tic'] == ticker].sort_values('date')
            if len(ticker_data) == self.lookback:
                # Calculate basic features
                returns = ticker_data['close'].pct_change().fillna(0).values
                volume_ma = ticker_data['volume'].rolling(window=10, min_periods=1).mean().values
                price_ma = ticker_data['close'].rolling(window=20, min_periods=1).mean().values

                # RSI calculation
                delta = ticker_data['close'].diff().fillna(0)
                gain = (delta.clip(lower=0)).rolling(window=14, min_periods=1).mean()
                loss = (-delta.clip(upper=0)).rolling(window=14, min_periods=1).mean()
                rs = gain / (loss + 1e-8)
                rsi = 100 - (100 / (1 + rs)).values

                # MACD
                exp1 = ticker_data['close'].ewm(span=12, adjust=False).mean()
                exp2 = ticker_data['close'].ewm(span=26, adjust=False).mean()
                macd = (exp1 - exp2).values

                # Volatility
                volatility = ticker_data['close'].pct_change().rolling(window=30, min_periods=1).std().fillna(0).values

                # Momentum
                momentum = ticker_data['close'].pct_change(periods=10).fillna(0).values

                # Combine the features (using only feature_dimension number of features)
                for j in range(self.lookback):
                    if self.feature_dimension >= 1: features[i, j, 0] = returns[j]
                    if self.feature_dimension >= 2: features[i, j, 1] = volume_ma[j]
                    if self.feature_dimension >= 3: features[i, j, 2] = price_ma[j]
                    if self.feature_dimension >= 4: features[i, j, 3] = rsi[j]
                    if self.feature_dimension >= 5: features[i, j, 4] = macd[j]
                    if self.feature_dimension >= 6: features[i, j, 5] = volatility[j]
                    if self.feature_dimension >= 7: features[i, j, 6] = momentum[j]

        self.recent_features = features
        print("Feature calculation complete!")

    def recommend_portfolio(self, amount_cad, profit_target_percentage=10):
        """Generate portfolio recommendations with profit tracking."""
        print(f"Generating recommendations for ${amount_cad} investment...")

        # Current date and future recommendation date
        current_date = pd.Timestamp.now()
        execution_date_str = current_date.strftime('%Y-%m-%d')
        recommendation_date = current_date + pd.DateOffset(days=7)
        recommendation_date_str = recommendation_date.strftime('%Y-%m-%d')

        # Use the model to predict allocations
        action, _ = self.model.predict(self.recent_features, deterministic=True)

        # Normalize allocations
        action = np.clip(action, 0, 1)
        action_sum = action.sum()
        if action_sum > 0:
            action /= action_sum
        else:
            action[0] = 1.0  # Set cash allocation to 100%

        # Allocate cash based on model recommendation
        allocations = {}
        cash_allocation = action[0] * amount_cad

        # Process allocations for each stock
        new_allocation_tickers = set()

        print("Processing stock allocations...")
        for i, ticker in enumerate(self.tickers):
            if i >= len(self.tickers) or i+1 >= len(action):
                continue

            allocation = action[i+1] * amount_cad

            # Only include stocks with positive allocations
            if allocation > 0 and ticker in self.latest_prices:
                price = self.latest_prices[ticker]
                shares = allocation / price if price > 0 else 0

                # Determine action type and profit
                if ticker in self.previous_allocations:
                    prev_shares = self.previous_allocations[ticker].get('shares', 0)

                    if shares > prev_shares:
                        action_type = "buy"
                        profit = allocation * (profit_target_percentage / 100)
                        profit_percentage = profit_target_percentage
                    elif shares < prev_shares:
                        action_type = "sell"
                        if ticker in self.purchase_history:
                            purchase_price = self.purchase_history[ticker].get('avg_price', price)
                            shares_sold = prev_shares - shares
                            profit = (price - purchase_price) * shares_sold
                            profit_percentage = ((price - purchase_price) / purchase_price * 100) if purchase_price > 0 else 0
                        else:
                            profit = 0
                            profit_percentage = 0
                    else:
                        action_type = "hold"
                        if ticker in self.purchase_history:
                            purchase_price = self.purchase_history[ticker].get('avg_price', price)
                            profit = (price - purchase_price) * shares
                            profit_percentage = ((price - purchase_price) / purchase_price * 100) if purchase_price > 0 else 0
                        else:
                            profit = 0
                            profit_percentage = 0
                else:
                    action_type = "buy"
                    profit = allocation * (profit_target_percentage / 100)
                    profit_percentage = profit_target_percentage

                # Add to new allocations
                new_allocation_tickers.add(ticker)

                # Update purchase history for buys
                if action_type == "buy":
                    if ticker not in self.purchase_history:
                        self.purchase_history[ticker] = {
                            'avg_price': price,
                            'total_shares': shares,
                            'initial_purchase_date': execution_date_str
                        }
                    else:
                        # Calculate new average purchase price
                        current_shares = self.purchase_history[ticker]['total_shares']
                        current_avg_price = self.purchase_history[ticker]['avg_price']

                        new_total_shares = current_shares + shares
                        if new_total_shares > 0:
                            new_avg_price = ((current_shares * current_avg_price) + (shares * price)) / new_total_shares
                            self.purchase_history[ticker]['avg_price'] = new_avg_price
                            self.purchase_history[ticker]['total_shares'] = new_total_shares

                # Store allocation details
                allocations[ticker] = {
                    "allocation_percent": float(action[i+1] * 100),
                    "allocation_cad": float(allocation),
                    "price_per_share": float(price),
                    "shares": float(shares),
                    "action": action_type,
                    "action_date": recommendation_date_str,
                    "profit": float(profit),
                    "profit_percentage": float(profit_percentage)
                }

        # Check for stocks that are being fully sold
        for ticker in self.previous_allocations:
            if ticker not in new_allocation_tickers:
                price = self.latest_prices.get(ticker, 0.0)
                prev_shares = self.previous_allocations[ticker].get('shares', 0)

                # Calculate profit from sale
                profit = 0.0
                profit_percentage = 0.0
                if ticker in self.purchase_history and prev_shares > 0:
                    purchase_price = self.purchase_history[ticker].get('avg_price', price)
                    profit = (price - purchase_price) * prev_shares
                    profit_percentage = ((price - purchase_price) / purchase_price * 100) if purchase_price > 0 else 0

                allocations[ticker] = {
                    "allocation_percent": 0.0,
                    "allocation_cad": 0.0,
                    "price_per_share": price,
                    "shares": 0.0,
                    "action": "sell",
                    "action_date": recommendation_date_str,
                    "profit": float(profit),
                    "profit_percentage": float(profit_percentage)
                }

        # Create action table
        action_table = []
        for ticker, details in allocations.items():
            action_table.append({
                "stock": ticker,
                "date": details["action_date"],
                "action": details["action"],
                "shares": details["shares"],
                "price": details["price_per_share"],
                "profit": details.get("profit", 0.0),
                "profit_percentage": details.get("profit_percentage", 0.0)
            })

        # Update previous allocations for next time
        self.previous_allocations = {
            ticker: details for ticker, details in allocations.items()
            if details["allocation_percent"] > 0
        }

        # Calculate total profit
        total_profit = sum(details.get("profit", 0.0) for details in allocations.values())
        total_profit_percentage = (total_profit / amount_cad * 100) if amount_cad > 0 else 0.0

        print(f"Recommendations generated! Expected profit: ${total_profit:.2f} ({total_profit_percentage:.2f}%)")

        # Return the portfolio recommendations
        return {
            "cash_percent": float(action[0] * 100),
            "cash_amount": float(cash_allocation),
            "stock_allocations": allocations,
            "total_amount": float(amount_cad),
            "total_profit": float(total_profit),
            "total_profit_percentage": float(total_profit_percentage),
            "model_date": str(self.latest_date),
            "execution_date": execution_date_str,
            "recommendation_date": recommendation_date_str,
            "action_table": action_table
        }

    def display_action_table(self, recommendation_result):
        """Display a formatted table of stock actions"""
        action_table = recommendation_result.get("action_table", [])

        if not action_table:
            print("No actions to display.")
            return

        # Print header
        print(f"\nStock Actions (Recommendations for {recommendation_result['recommendation_date']}):")
        print(f"{'Stock':<8} {'Date':<12} {'Action':<6} {'Shares':<10} {'Price':<10} {'Profit':<12} {'Profit %':<10}")
        print("-" * 80)

        # Print each action
        for item in action_table:
            print(f"{item['stock']:<8} {item['date']:<12} {item['action']:<6} {item['shares']:<10.2f} ${item['price']:<9.2f} ${item['profit']:<11.2f} {item['profit_percentage']:<9.2f}%")

        # Print summary
        total_profit = recommendation_result.get("total_profit", 0.0)
        total_profit_percentage = recommendation_result.get("total_profit_percentage", 0.0)
        print("-" * 80)
        print(f"Total Portfolio Profit: ${total_profit:.2f} ({total_profit_percentage:.2f}%)")

    def export_action_table(self, recommendation_result, filename=None):
        """Export action table to CSV and print the data"""
        action_table = recommendation_result.get("action_table", [])

        if not action_table:
            print("No actions to export.")
            return

        if filename is None:
            filename = f"portfolio_actions_{recommendation_result['recommendation_date']}.csv"

        # Convert to DataFrame
        df = pd.DataFrame(action_table)

        # Export to CSV
        df.to_csv(filename, index=False)

        # Print the DataFrame
        print(f"\nAction Table Results:")
        print("-" * 100)
        print(df.to_string(index=False))
        print("-" * 100)

        # Print summary statistics
        total_profit = recommendation_result.get("total_profit", 0.0)
        total_profit_percentage = recommendation_result.get("total_profit_percentage", 0.0)
        print(f"Total Portfolio Profit: ${total_profit:.2f} ({total_profit_percentage:.2f}%)")
        print(f"Action table exported to {filename}")

    def simulate_future_recommendations(self, amount_cad, future_dates=30, price_change_range=(-0.15, 0.25)):
        """
        Simulate portfolio recommendations over multiple future dates to show buy and sell actions

        Args:
            amount_cad: Initial investment amount
            future_dates: Number of days to simulate into the future
            price_change_range: Range of random price changes (min, max) as percentages

        Returns:
            List of portfolio recommendation results for each date
        """
        print(f"Simulating portfolio evolution over {future_dates} days...")

        # Store results for each date
        all_recommendations = []

        # Initial portfolio recommendation
        initial_portfolio = self.recommend_portfolio(amount_cad=amount_cad)
        all_recommendations.append(initial_portfolio)

        # Clone the latest prices for simulation
        simulated_prices = self.latest_prices.copy()
        original_prices = self.latest_prices.copy()

        # Track cumulative profit
        cumulative_profit = initial_portfolio["total_profit"]

        # Simulate for future dates
        for day in range(1, future_dates + 1):
            # Simulate price changes
            for ticker in simulated_prices:
                # Generate random price change within range
                pct_change = random.uniform(price_change_range[0], price_change_range[1])

                # Apply the change
                simulated_prices[ticker] = simulated_prices[ticker] * (1 + pct_change)

                # Introduce some trend based on previous recommendations
                # Stocks that performed well tend to continue doing well
                if ticker in self.purchase_history:
                    purchase_price = self.purchase_history[ticker].get('avg_price', 0)
                    current_price = simulated_prices[ticker]

                    # If the stock is doing well, slightly bias toward continued growth
                    if current_price > purchase_price * 1.1:  # More than 10% gain
                        simulated_prices[ticker] *= 1.01  # Small positive bias

                    # If the stock is doing poorly, slightly increase chance of recovery or further decline
                    elif current_price < purchase_price * 0.9:  # More than 10% loss
                        # Randomly decide if it will recover or decline more
                        if random.random() > 0.5:
                            simulated_prices[ticker] *= 1.02  # Small recovery
                        else:
                            simulated_prices[ticker] *= 0.98  # Further decline

            # Save original prices
            original_latest_prices = self.latest_prices

            # Temporarily replace prices with simulated ones
            self.latest_prices = simulated_prices

            # Create a date for this simulation
            simulation_date = pd.Timestamp.now() + pd.DateOffset(days=day)

            # Generate recommendation for this date
            print(f"\nDay {day} - Simulating for {simulation_date.strftime('%Y-%m-%d')}...")
            try:
                portfolio = self.recommend_portfolio(
                    amount_cad=amount_cad,
                    profit_target_percentage=10
                )

                # Add to results
                all_recommendations.append(portfolio)

                # Update cumulative profit
                day_profit = portfolio["total_profit"]
                cumulative_profit += day_profit

                print(f"Day {day} profit: ${day_profit:.2f}, Cumulative: ${cumulative_profit:.2f}")

            except Exception as e:
                print(f"Error on day {day}: {e}")

            # Restore original prices
            self.latest_prices = original_latest_prices

        print(f"\nSimulation complete. Generated {len(all_recommendations)} recommendations.")
        return all_recommendations

    def display_sell_signals(self, recommendation_results):
        """Display all sell signals from the recommendations"""
        print("\n=== SELL SIGNALS DETECTED ===")
        print(f"{'Day':<5} {'Date':<12} {'Stock':<8} {'Price':<10} {'Profit':<12} {'Profit %':<10}")
        print("-" * 70)

        day = 0
        sell_signals_found = False

        for result in recommendation_results:
            day += 1
            date = result['recommendation_date']

            for ticker, details in result['stock_allocations'].items():
                if details['action'] == 'sell':
                    sell_signals_found = True
                    print(f"{day:<5} {date:<12} {ticker:<8} ${details['price_per_share']:<9.2f} ${details['profit']:<11.2f} {details['profit_percentage']:<9.2f}%")

        if not sell_signals_found:
            print("No sell signals detected in the simulation period.")

    def generate_buy_sell_report(self, recommendation_results, output_file="buy_sell_report.csv"):
        """Generate a comprehensive buy/sell report from simulation results"""
        # Prepare data
        report_data = []

        day = 0
        for result in recommendation_results:
            day += 1
            date = result['recommendation_date']

            for ticker, details in result['stock_allocations'].items():
                if details['action'] in ['buy', 'sell']:  # Include only buy and sell actions
                    report_data.append({
                        'day': day,
                        'date': date,
                        'stock': ticker,
                        'action': details['action'],
                        'shares': details['shares'],
                        'price': details['price_per_share'],
                        'allocation_cad': details['allocation_cad'],
                        'profit': details['profit'],
                        'profit_percentage': details['profit_percentage']
                    })

        # Convert to DataFrame and save
        df = pd.DataFrame(report_data)

        # Save to CSV
        df.to_csv(output_file, index=False)

        # Create summary
        buy_actions = df[df['action'] == 'buy']
        sell_actions = df[df['action'] == 'sell']

        print(f"\n=== BUY/SELL REPORT SUMMARY ===")
        print(f"Total Buy Actions: {len(buy_actions)}")
        print(f"Total Sell Actions: {len(sell_actions)}")

        if len(sell_actions) > 0:
            print(f"Total Profit from Sells: ${sell_actions['profit'].sum():.2f}")

        print(f"Report saved to: {output_file}")

        return df

    def plot_sell_recommendations(self, recommendation_results):
        """Plot sell recommendations by date"""
        # Collect sell actions
        sell_data = []

        day = 0
        for result in recommendation_results:
            day += 1
            date = result['recommendation_date']

            day_sells = 0
            day_profit = 0

            for ticker, details in result['stock_allocations'].items():
                if details['action'] == 'sell':
                    day_sells += 1
                    day_profit += details['profit']

            if day_sells > 0:
                sell_data.append({
                    'day': day,
                    'date': date,
                    'count': day_sells,
                    'profit': day_profit
                })

        # Convert to DataFrame
        df = pd.DataFrame(sell_data)

        if len(df) == 0:
            print("No sell recommendations to plot.")
            return

        # Create plot
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

        # Plot sell counts
        ax1.bar(df['day'], df['count'], color='red', alpha=0.7)
        ax1.set_title('Number of Sell Recommendations by Day')
        ax1.set_ylabel('Count')
        ax1.grid(True, linestyle='--', alpha=0.7)

        # Plot profit
        ax2.bar(df['day'], df['profit'], color='green', alpha=0.7)
        ax2.set_title('Profit from Sell Recommendations by Day')
        ax2.set_xlabel('Day')
        ax2.set_ylabel('Profit ($)')
        ax2.grid(True, linestyle='--', alpha=0.7)

        plt.tight_layout()
        plt.show()

        return df

Mounted at /content/drive


In [3]:
def simulate_future_recommendations_with_varying_dates(self, amount_cad, future_days=30, price_change_range=(-0.15, 0.25)):
    """
    Simulate portfolio recommendations with realistic dates based on optimal entry/exit points

    Args:
        amount_cad: Initial investment amount
        future_days: Number of days to simulate into the future
        price_change_range: Range of random price changes (min, max) as percentages

    Returns:
        Dictionary containing portfolio recommendations with varying dates
    """
    print(f"Simulating portfolio evolution over {future_days} days with realistic date-based recommendations...")

    # Initial portfolio recommendation
    initial_portfolio = self.recommend_portfolio(amount_cad=amount_cad)

    # Clone the latest prices for simulation
    simulated_prices = {ticker: self.latest_prices[ticker] for ticker in self.latest_prices}
    price_history = {ticker: [self.latest_prices[ticker]] for ticker in self.latest_prices}

    # Store action recommendations with dates
    date_based_recommendations = {
        "buys": [],
        "sells": [],
        "holds": []
    }

    # Simulation start date
    current_date = pd.Timestamp.now()
    date_range = [current_date + pd.DateOffset(days=i) for i in range(1, future_days + 1)]

    # Track price trends for each stock to identify optimal points
    ticker_trends = {ticker: {
        'trend': 'neutral',   # 'up', 'down', 'neutral'
        'days_in_trend': 0,
        'price_change': 0.0,
        'last_action_day': 0,
        'buy_opportunity': False,
        'sell_opportunity': False,
        'avg_price': self.latest_prices[ticker],
        'stop_loss': self.latest_prices[ticker] * 0.85,  # 15% stop loss
        'take_profit': self.latest_prices[ticker] * 1.2,  # 20% take profit
    } for ticker in self.latest_prices}

    # Establish initial positions based on initial recommendation
    positions = {}
    for ticker, details in initial_portfolio['stock_allocations'].items():
        if details['action'] == 'buy' and details['shares'] > 0:
            positions[ticker] = {
                'shares': details['shares'],
                'avg_price': details['price_per_share'],
                'cost_basis': details['allocation_cad'],
                'buy_date': current_date,
                'stop_loss': details['price_per_share'] * 0.85,
                'take_profit': details['price_per_share'] * 1.2,
            }

            # Record initial buys
            date_based_recommendations['buys'].append({
                'stock': ticker,
                'date': current_date.strftime('%Y-%m-%d'),
                'action': 'buy',
                'shares': details['shares'],
                'price': details['price_per_share'],
                'profit': 0.0,
                'profit_percentage': 0.0,
                'reason': 'Initial portfolio allocation'
            })

    # Simulate price evolution and identify trading opportunities
    cumulative_profit = 0.0

    print(f"Starting simulation on {current_date.strftime('%Y-%m-%d')}...")

    for day in range(1, future_days + 1):
        # Current date in simulation
        sim_date = date_range[day-1]

        # Simulate price changes
        for ticker in simulated_prices:
            # Generate random price change within range
            # Make price changes less random and more trend-based

            # Use current trend to influence next price move (momentum)
            trend_factor = 0
            if ticker in ticker_trends:
                if ticker_trends[ticker]['trend'] == 'up':
                    trend_factor = 0.05  # More likely to continue up
                elif ticker_trends[ticker]['trend'] == 'down':
                    trend_factor = -0.05  # More likely to continue down

            # Base random component
            pct_change = random.uniform(price_change_range[0], price_change_range[1])

            # Add trend factor
            pct_change += trend_factor

            # Apply the change
            old_price = simulated_prices[ticker]
            new_price = old_price * (1 + pct_change)
            simulated_prices[ticker] = new_price

            # Update price history
            if ticker in price_history:
                price_history[ticker].append(new_price)

            # Update trend information
            if ticker in ticker_trends:
                # Calculate daily change
                daily_change = (new_price / old_price) - 1

                # Update price change from starting point
                ticker_trends[ticker]['price_change'] = (new_price / price_history[ticker][0]) - 1

                # Detect trend
                old_trend = ticker_trends[ticker]['trend']

                if daily_change > 0.02:  # 2% up day
                    if old_trend == 'up':
                        ticker_trends[ticker]['days_in_trend'] += 1
                    else:
                        ticker_trends[ticker]['trend'] = 'up'
                        ticker_trends[ticker]['days_in_trend'] = 1
                elif daily_change < -0.02:  # 2% down day
                    if old_trend == 'down':
                        ticker_trends[ticker]['days_in_trend'] += 1
                    else:
                        ticker_trends[ticker]['trend'] = 'down'
                        ticker_trends[ticker]['days_in_trend'] = 1
                else:
                    # Neutral day, continue previous trend but don't increment
                    pass

                # Buy opportunity:
                # 1. Stock has been going down for a while but starts to recover
                # 2. Stock is in strong uptrend
                if (old_trend == 'down' and daily_change > 0.04) or \
                   (old_trend == 'up' and ticker_trends[ticker]['days_in_trend'] >= 3 and
                    ticker_trends[ticker]['days_in_trend'] - ticker_trends[ticker]['last_action_day'] >= 3):
                    ticker_trends[ticker]['buy_opportunity'] = True
                else:
                    ticker_trends[ticker]['buy_opportunity'] = False

                # Sell opportunity:
                # 1. Stock has been going up but now turning down
                # 2. Stock hits take profit level
                # 3. Stock hits stop loss level
                # 4. Stock in prolonged downtrend
                if ticker in positions and positions[ticker]['shares'] > 0:
                    if (old_trend == 'up' and daily_change < -0.03) or \
                       (new_price >= positions[ticker]['take_profit']) or \
                       (new_price <= positions[ticker]['stop_loss']) or \
                       (old_trend == 'down' and ticker_trends[ticker]['days_in_trend'] >= 5):
                        ticker_trends[ticker]['sell_opportunity'] = True
                    else:
                        ticker_trends[ticker]['sell_opportunity'] = False

        # Generate buy/sell/hold recommendations based on trends
        actions_today = []

        for ticker in ticker_trends:
            # Buy logic
            if ticker_trends[ticker]['buy_opportunity'] and ticker not in positions:
                # Calculate how many shares we can buy
                price = simulated_prices[ticker]
                allocation = amount_cad * 0.05  # Allocate 5% of portfolio to each position
                shares = allocation / price if price > 0 else 0

                if shares > 0:
                    # Record the buy recommendation
                    buy_reason = ""
                    if ticker_trends[ticker]['trend'] == 'down' and ticker_trends[ticker]['price_change'] < -0.1:
                        buy_reason = "Potential bottom (down trend reversal)"
                    elif ticker_trends[ticker]['trend'] == 'up' and ticker_trends[ticker]['days_in_trend'] >= 3:
                        buy_reason = "Strong uptrend continuation"
                    else:
                        buy_reason = "Technical buy signal"

                    date_based_recommendations['buys'].append({
                        'stock': ticker,
                        'date': sim_date.strftime('%Y-%m-%d'),
                        'action': 'buy',
                        'shares': shares,
                        'price': price,
                        'profit': 0.0,
                        'profit_percentage': 0.0,
                        'reason': buy_reason
                    })

                    # Record the position
                    positions[ticker] = {
                        'shares': shares,
                        'avg_price': price,
                        'cost_basis': allocation,
                        'buy_date': sim_date,
                        'stop_loss': price * 0.85,
                        'take_profit': price * 1.2,
                    }

                    # Update last action day
                    ticker_trends[ticker]['last_action_day'] = ticker_trends[ticker]['days_in_trend']

                    actions_today.append(f"BUY {ticker}: {shares:.2f} shares @ ${price:.2f} - {buy_reason}")

            # Sell logic
            elif ticker in positions and ticker_trends[ticker]['sell_opportunity']:
                price = simulated_prices[ticker]
                shares = positions[ticker]['shares']
                avg_price = positions[ticker]['avg_price']
                cost_basis = positions[ticker]['cost_basis']

                if shares > 0:
                    # Calculate profit
                    profit = (price - avg_price) * shares
                    profit_percentage = ((price / avg_price) - 1) * 100

                    # Record the sell recommendation
                    sell_reason = ""
                    if price >= positions[ticker]['take_profit']:
                        sell_reason = f"Take profit target reached (+20%)"
                    elif price <= positions[ticker]['stop_loss']:
                        sell_reason = f"Stop loss triggered (-15%)"
                    elif ticker_trends[ticker]['trend'] == 'up' and ticker_trends[ticker]['price_change'] > 0.15:
                        sell_reason = "Taking profits after strong rally"
                    elif ticker_trends[ticker]['trend'] == 'down' and ticker_trends[ticker]['days_in_trend'] >= 5:
                        sell_reason = "Cutting losses in extended downtrend"
                    else:
                        sell_reason = "Technical sell signal"

                    date_based_recommendations['sells'].append({
                        'stock': ticker,
                        'date': sim_date.strftime('%Y-%m-%d'),
                        'action': 'sell',
                        'shares': shares,
                        'price': price,
                        'profit': profit,
                        'profit_percentage': profit_percentage,
                        'reason': sell_reason
                    })

                    # Update cumulative profit
                    cumulative_profit += profit

                    # Mark position as closed
                    positions[ticker]['shares'] = 0

                    # Update last action day
                    ticker_trends[ticker]['last_action_day'] = ticker_trends[ticker]['days_in_trend']

                    actions_today.append(f"SELL {ticker}: {shares:.2f} shares @ ${price:.2f} - Profit: ${profit:.2f} ({profit_percentage:.2f}%) - {sell_reason}")

            # Hold logic - record significant unrealized profits or losses
            elif ticker in positions and positions[ticker]['shares'] > 0:
                price = simulated_prices[ticker]
                shares = positions[ticker]['shares']
                avg_price = positions[ticker]['avg_price']
                days_held = (sim_date - positions[ticker]['buy_date']).days

                # Only record holds every 5 days or on significant price movements
                if days_held > 0 and (days_held % 5 == 0 or abs((price/avg_price - 1) * 100) > 10):
                    unrealized_profit = (price - avg_price) * shares
                    unrealized_pct = ((price / avg_price) - 1) * 100

                    hold_reason = ""
                    if unrealized_pct > 15:
                        hold_reason = "Holding winning position, approaching target"
                    elif unrealized_pct < -10:
                        hold_reason = "Holding through drawdown, monitoring for reversal"
                    else:
                        hold_reason = "Maintaining position, no exit signal"

                    date_based_recommendations['holds'].append({
                        'stock': ticker,
                        'date': sim_date.strftime('%Y-%m-%d'),
                        'action': 'hold',
                        'shares': shares,
                        'price': price,
                        'days_held': days_held,
                        'unrealized_profit': unrealized_profit,
                        'unrealized_percentage': unrealized_pct,
                        'reason': hold_reason
                    })

                    actions_today.append(f"HOLD {ticker}: {shares:.2f} shares, ${unrealized_profit:.2f} unrealized ({unrealized_pct:.2f}%) - {hold_reason}")

        # Print daily summary if there were actions
        if actions_today:
            print(f"\nDay {day} - {sim_date.strftime('%Y-%m-%d')} - Actions:")
            for action in actions_today:
                print(f"  {action}")

    print(f"\nSimulation complete. Cumulative realized profit: ${cumulative_profit:.2f}")
    print(f"Total buy recommendations: {len(date_based_recommendations['buys'])}")
    print(f"Total sell recommendations: {len(date_based_recommendations['sells'])}")
    print(f"Total hold notifications: {len(date_based_recommendations['holds'])}")

    return date_based_recommendations


def display_date_based_recommendations(self, recommendations):
    """
    Display date-based recommendations in a chronological format

    Args:
        recommendations: Output from simulate_future_recommendations_with_varying_dates
    """
    # Combine all recommendations
    all_recs = []

    for buy in recommendations['buys']:
        all_recs.append(buy)

    for sell in recommendations['sells']:
        all_recs.append(sell)

    for hold in recommendations['holds']:
        all_recs.append(hold)

    # Sort by date
    all_recs.sort(key=lambda x: x['date'])

    # Display chronologically
    print("\n=== CHRONOLOGICAL TRADING RECOMMENDATIONS ===")
    print(f"{'Date':<12} {'Stock':<8} {'Action':<6} {'Shares':<10} {'Price':<10} {'Profit':<12} {'Profit %':<10} {'Reason'}")
    print("-" * 120)

    current_date = None

    for rec in all_recs:
        if current_date != rec['date']:
            current_date = rec['date']
            print(f"\n--- {current_date} ---")

        # Handle different action types slightly differently
        if rec['action'] == 'hold':
            print(f"{'':<12} {rec['stock']:<8} {rec['action']:<6} {rec['shares']:<10.2f} ${rec['price']:<9.2f} ${rec['unrealized_profit']:<11.2f} {rec['unrealized_percentage']:<9.2f}% {rec['reason']}")
        else:
            print(f"{'':<12} {rec['stock']:<8} {rec['action']:<6} {rec['shares']:<10.2f} ${rec['price']:<9.2f} ${rec.get('profit', 0.0):<11.2f} {rec.get('profit_percentage', 0.0):<9.2f}% {rec['reason']}")

    # Calculate profitability stats
    total_buys = len(recommendations['buys'])
    total_sells = len(recommendations['sells'])
    total_profit = sum(sell.get('profit', 0.0) for sell in recommendations['sells'])
    winning_trades = sum(1 for sell in recommendations['sells'] if sell.get('profit', 0.0) > 0)
    losing_trades = sum(1 for sell in recommendations['sells'] if sell.get('profit', 0.0) <= 0)

    win_rate = winning_trades / total_sells * 100 if total_sells > 0 else 0
    avg_win = sum(sell.get('profit', 0.0) for sell in recommendations['sells'] if sell.get('profit', 0.0) > 0) / winning_trades if winning_trades > 0 else 0
    avg_loss = sum(sell.get('profit', 0.0) for sell in recommendations['sells'] if sell.get('profit', 0.0) <= 0) / losing_trades if losing_trades > 0 else 0

    print("\n=== TRADING PERFORMANCE SUMMARY ===")
    print(f"Total Buy Recommendations: {total_buys}")
    print(f"Total Sell Recommendations: {total_sells}")
    print(f"Total Realized Profit: ${total_profit:.2f}")
    print(f"Win Rate: {win_rate:.2f}%")
    print(f"Average Win: ${avg_win:.2f}")
    print(f"Average Loss: ${avg_loss:.2f}")
    if avg_loss != 0:
        print(f"Risk-Reward Ratio: {abs(avg_win/avg_loss):.2f}")


def export_date_based_recommendations(self, recommendations, filename="date_based_recommendations.csv"):
    """
    Export date-based recommendations to a CSV file

    Args:
        recommendations: Output from simulate_future_recommendations_with_varying_dates
        filename: Output filename
    """
    # Combine all recommendations
    all_recs = []

    for buy in recommendations['buys']:
        rec = buy.copy()
        all_recs.append(rec)

    for sell in recommendations['sells']:
        rec = sell.copy()
        all_recs.append(rec)

    for hold in recommendations['holds']:
        rec = hold.copy()
        # Rename fields to match the others
        if 'unrealized_profit' in rec:
            rec['profit'] = rec['unrealized_profit']
            del rec['unrealized_profit']
        if 'unrealized_percentage' in rec:
            rec['profit_percentage'] = rec['unrealized_percentage']
            del rec['unrealized_percentage']
        all_recs.append(rec)

    # Sort by date
    all_recs.sort(key=lambda x: x['date'])

    # Convert to DataFrame
    df = pd.DataFrame(all_recs)

    # Save to CSV
    df.to_csv(filename, index=False)

    print(f"\nAll recommendations exported to {filename}")

    # Also export a summary by day
    daily_summary = {}
    for rec in all_recs:
        date = rec['date']
        action = rec['action']

        if date not in daily_summary:
            daily_summary[date] = {'buys': 0, 'sells': 0, 'holds': 0, 'profit': 0.0}

        daily_summary[date][action + 's'] += 1
        if action == 'sell':
            daily_summary[date]['profit'] += rec.get('profit', 0.0)

    # Convert to DataFrame
    df_summary = pd.DataFrame([
        {'date': date, 'buys': data['buys'], 'sells': data['sells'],
         'holds': data['holds'], 'daily_profit': data['profit']}
        for date, data in daily_summary.items()
    ])

    # Sort by date
    df_summary = df_summary.sort_values('date')

    # Add cumulative profit
    df_summary['cumulative_profit'] = df_summary['daily_profit'].cumsum()

    # Save to CSV
    summary_filename = "daily_trading_summary.csv"
    df_summary.to_csv(summary_filename, index=False)

    print(f"Daily trading summary exported to {summary_filename}")

    # Return both DataFrames
    return df, df_summary


def plot_trading_summary(self, recommendations):
    """
    Create visualizations of the trading recommendations and performance

    Args:
        recommendations: Output from simulate_future_recommendations_with_varying_dates
    """
    # Combine all recommendations
    all_recs = []

    for buy in recommendations['buys']:
        rec = buy.copy()
        rec['profit'] = 0.0
        rec['profit_percentage'] = 0.0
        all_recs.append(rec)

    for sell in recommendations['sells']:
        all_recs.append(sell)

    # Sort by date
    all_recs.sort(key=lambda x: x['date'])

    # Create a DataFrame
    df = pd.DataFrame(all_recs)

    # Convert date to datetime
    df['date'] = pd.to_datetime(df['date'])

    # Create daily summary
    daily_summary = df.groupby(['date', 'action']).agg({
        'stock': 'count',
        'profit': 'sum'
    }).reset_index()

    # Pivot to get buys and sells by day
    pivoted = daily_summary.pivot_table(
        index='date',
        columns='action',
        values='stock',
        fill_value=0
    ).reset_index()

    if 'sell' not in pivoted.columns:
        pivoted['sell'] = 0
    if 'buy' not in pivoted.columns:
        pivoted['buy'] = 0

    # Calculate daily and cumulative profit
    daily_profit = daily_summary[daily_summary['action'] == 'sell'].groupby('date')['profit'].sum().reset_index()
    daily_profit['cumulative_profit'] = daily_profit['profit'].cumsum()

    # Create plots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    # Plot buy/sell counts
    ax1.bar(pivoted['date'], pivoted['buy'], color='green', alpha=0.7, label='Buys')
    ax1.bar(pivoted['date'], pivoted['sell'], color='red', alpha=0.7, label='Sells')
    ax1.set_title('Number of Buy/Sell Recommendations by Date')
    ax1.set_ylabel('Count')
    ax1.legend()
    ax1.grid(True, linestyle='--', alpha=0.7)

    # Plot profit
    if not daily_profit.empty:
        ax2.bar(daily_profit['date'], daily_profit['profit'], color='blue', alpha=0.7)
        ax2.set_title('Daily Profit from Sells')
        ax2.set_xlabel('Date')
        ax2.set_ylabel('Profit ($)')
        ax2.grid(True, linestyle='--', alpha=0.7)

        # Add cumulative profit line
        ax3 = ax2.twinx()
        ax3.plot(daily_profit['date'], daily_profit['cumulative_profit'], 'r-', label='Cumulative Profit')
        ax3.set_ylabel('Cumulative Profit ($)')
        ax3.legend(loc='upper right')

    plt.tight_layout()
    plt.show()

    # Create performance visualization by stock
    stock_performance = df[df['action'] == 'sell'].groupby('stock').agg({
        'profit': 'sum',
        'profit_percentage': 'mean'
    }).reset_index()

    # Sort by total profit
    stock_performance = stock_performance.sort_values('profit', ascending=False)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # Plot total profit by stock
    ax1.bar(stock_performance['stock'], stock_performance['profit'], color='blue', alpha=0.7)
    ax1.set_title('Total Profit by Stock')
    ax1.set_xlabel('Stock')
    ax1.set_ylabel('Total Profit ($)')
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax1.tick_params(axis='x', rotation=90)

    # Plot average profit percentage by stock
    ax2.bar(stock_performance['stock'], stock_performance['profit_percentage'], color='green', alpha=0.7)
    ax2.set_title('Average Profit Percentage by Stock')
    ax2.set_xlabel('Stock')
    ax2.set_ylabel('Average Profit (%)')
    ax2.grid(True, linestyle='--', alpha=0.7)
    ax2.tick_params(axis='x', rotation=90)

    plt.tight_layout()
    plt.show()

In [4]:
# Define the function with consistent parameter naming
def simulate_future_recommendations_with_realistic_profits(self, amount_cad, future_days=30, price_change_range=(-0.15, 0.25), risk_profile='medium'):
    """
    Simulate portfolio recommendations with realistic dates and profit calculations

    Args:
        amount_cad: Initial investment amount
        future_days: Number of days to simulate into the future
        price_change_range: Range of random price changes (min, max) as percentages
        risk_profile: Risk profile to use ('low', 'medium', 'high')

    Returns:
        Dictionary containing portfolio recommendations with varying dates and realistic profits,
        final portfolio value, and cumulative realized profit
    """
    # Set up risk profiles for different risk levels
    RISK_PROFILES = {
        'low': {
            'stop_loss_pct': 0.10,    # 10% stop loss
            'take_profit_pct': 0.15,   # 15% take profit
            'max_position_pct': 0.03,  # max 3% in one position
            'cash_reserve_pct': 0.30   # keep 30% in cash
        },
        'medium': {
            'stop_loss_pct': 0.15,    # 15% stop loss
            'take_profit_pct': 0.20,   # 20% take profit
            'max_position_pct': 0.05,  # max 5% in one position
            'cash_reserve_pct': 0.25   # keep 25% in cash
        },
        'high': {
            'stop_loss_pct': 0.20,    # 20% stop loss
            'take_profit_pct': 0.30,   # 30% take profit
            'max_position_pct': 0.08,  # max 8% in one position
            'cash_reserve_pct': 0.15   # keep 15% in cash
        }
    }

    # Get risk parameters based on selected profile
    risk_config = RISK_PROFILES.get(risk_profile, RISK_PROFILES['medium'])
    stop_loss_factor = 1 - risk_config['stop_loss_pct']
    take_profit_factor = 1 + risk_config['take_profit_pct']
    max_position_pct = risk_config['max_position_pct']
    cash_reserve_pct = risk_config['cash_reserve_pct']

    print(f"Using {risk_profile} risk profile:")
    print(f"  - Stop loss: {risk_config['stop_loss_pct']*100:.0f}%")
    print(f"  - Take profit: {risk_config['take_profit_pct']*100:.0f}%")
    print(f"  - Max position size: {risk_config['max_position_pct']*100:.0f}% of portfolio")
    print(f"  - Cash reserve: {risk_config['cash_reserve_pct']*100:.0f}% of available cash")
    print(f"Simulating portfolio evolution over {future_days} days with realistic profit calculations...")

    # Initial portfolio recommendation
    initial_portfolio = self.recommend_portfolio(amount_cad=amount_cad)

    # Clone the latest prices for simulation
    simulated_prices = {ticker: self.latest_prices[ticker] for ticker in self.latest_prices}
    price_history = {ticker: [self.latest_prices[ticker]] for ticker in self.latest_prices}

    # Store action recommendations with dates
    date_based_recommendations = {
        "buys": [],
        "sells": [],
        "holds": []
    }

    # Simulation start date
    current_date = pd.Timestamp.now()
    date_range = [current_date + pd.DateOffset(days=i) for i in range(1, future_days + 1)]

    # Track price trends for each stock to identify optimal points
    ticker_trends = {ticker: {
        'trend': 'neutral',   # 'up', 'down', 'neutral'
        'days_in_trend': 0,
        'price_change': 0.0,
        'last_action_day': 0,
        'buy_opportunity': False,
        'sell_opportunity': False,
        'avg_price': self.latest_prices[ticker],
        'stop_loss': self.latest_prices[ticker] * stop_loss_factor,
        'take_profit': self.latest_prices[ticker] * take_profit_factor,
    } for ticker in self.latest_prices}

    # Establish initial positions based on initial recommendation
    positions = {}
    remaining_cash = amount_cad

    for ticker, details in initial_portfolio['stock_allocations'].items():
        if details['action'] == 'buy' and details['shares'] > 0:
            # Calculate the actual cost including transaction fees
            shares = details['shares']
            price = details['price_per_share']
            transaction_amount = shares * price

            # Apply transaction fee (typically 0.1% to 0.5%)
            transaction_fee = transaction_amount * 0.001  # 0.1% fee
            total_cost = transaction_amount + transaction_fee

            # Update remaining cash
            if total_cost <= remaining_cash:
                remaining_cash -= total_cost

                positions[ticker] = {
                    'shares': shares,
                    'avg_price': price,
                    'cost_basis': total_cost,
                    'transaction_fee': transaction_fee,
                    'buy_date': current_date,
                    'stop_loss': price * stop_loss_factor,
                    'take_profit': price * take_profit_factor,
                }

                # Record initial buys
                date_based_recommendations['buys'].append({
                    'stock': ticker,
                    'date': current_date.strftime('%Y-%m-%d'),
                    'action': 'buy',
                    'shares': shares,
                    'price': price,
                    'transaction_fee': transaction_fee,
                    'total_cost': total_cost,
                    'profit': 0.0,
                    'profit_percentage': 0.0,
                    'reason': 'Initial portfolio allocation'
                })

    # Simulate price evolution and identify trading opportunities
    cumulative_profit = 0.0

    print(f"Starting simulation on {current_date.strftime('%Y-%m-%d')} with ${remaining_cash:.2f} remaining cash...")

    for day in range(1, future_days + 1):
        # Current date in simulation
        sim_date = date_range[day-1]

        # Simulate price changes
        # Apply more realistic market behavior:
        # 1. Market-wide factors affect all stocks (systematic risk)
        # 2. Individual stock factors (idiosyncratic risk)
        # 3. Sector correlation (stocks in same sector move together)

        # Simulate market-wide movement (affects all stocks)
        market_move = random.normalvariate(0.0002, 0.01)  # Slight positive bias with volatility

        # Sector correlations (simplified)
        sectors = {
            'tech': ['SHOP.TO', 'BB.TO', 'OTEX.TO'],
            'energy': ['ENB.TO', 'TRP.TO', 'CNQ.TO', 'SU.TO', 'CVE.TO', 'BTE.TO', 'GEI.TO'],
            'materials': ['ABX.TO', 'FNV.TO', 'K.TO', 'AEM.TO', 'NXE.TO'],
            'financials': ['RY.TO', 'TD.TO', 'BNS.TO', 'NA.TO', 'CM.TO'],
            'utilities': ['FTS.TO', 'EMA.TO', 'H.TO'],
            'real_estate': ['REI.UN.TO', 'HR.UN.TO'],
            'telecom': ['BCE.TO', 'T.TO', 'RCI-B.TO'],
            'consumer': ['ATD.TO', 'L.TO', 'MRU.TO', 'QSR.TO'],
            'healthcare': ['SIA.TO', 'DOC.TO'],
            'industrial': ['CNR.TO', 'CP.TO', 'WCN.TO']
        }

        # Generate sector moves
        sector_moves = {sector: random.normalvariate(market_move, 0.008) for sector in sectors}

        # Process each stock
        for ticker in simulated_prices:
            # Find which sector the stock belongs to
            stock_sector = None
            for sector, stocks in sectors.items():
                if ticker in stocks:
                    stock_sector = sector
                    break

            # Base movement components
            sector_component = sector_moves.get(stock_sector, 0) if stock_sector else 0
            stock_specific_component = random.normalvariate(0, 0.02)  # Stock-specific volatility

            # Apply trend continuity (momentum effect)
            trend_component = 0
            if ticker in ticker_trends:
                if ticker_trends[ticker]['trend'] == 'up':
                    trend_component = random.uniform(0, 0.01)  # More likely to continue up
                elif ticker_trends[ticker]['trend'] == 'down':
                    trend_component = random.uniform(-0.01, 0)  # More likely to continue down

            # Combine components for final price change
            pct_change = market_move + sector_component + stock_specific_component + trend_component

            # Apply the change
            old_price = simulated_prices[ticker]
            new_price = old_price * (1 + pct_change)
            simulated_prices[ticker] = new_price

            # Update price history
            if ticker in price_history:
                price_history[ticker].append(new_price)

            # Update trend information
            if ticker in ticker_trends:
                # Calculate daily change
                daily_change = (new_price / old_price) - 1

                # Update price change from starting point
                ticker_trends[ticker]['price_change'] = (new_price / price_history[ticker][0]) - 1

                # Detect trend
                old_trend = ticker_trends[ticker]['trend']

                if daily_change > 0.02:  # 2% up day
                    if old_trend == 'up':
                        ticker_trends[ticker]['days_in_trend'] += 1
                    else:
                        ticker_trends[ticker]['trend'] = 'up'
                        ticker_trends[ticker]['days_in_trend'] = 1
                elif daily_change < -0.02:  # 2% down day
                    if old_trend == 'down':
                        ticker_trends[ticker]['days_in_trend'] += 1
                    else:
                        ticker_trends[ticker]['trend'] = 'down'
                        ticker_trends[ticker]['days_in_trend'] = 1
                else:
                    # Neutral day, continue previous trend but don't increment
                    pass

                # Buy opportunity:
                # 1. Stock has been going down for a while but starts to recover
                # 2. Stock is in strong uptrend
                if (old_trend == 'down' and daily_change > 0.04) or \
                   (old_trend == 'up' and ticker_trends[ticker]['days_in_trend'] >= 3 and
                    ticker_trends[ticker]['days_in_trend'] - ticker_trends[ticker]['last_action_day'] >= 3):
                    ticker_trends[ticker]['buy_opportunity'] = True
                else:
                    ticker_trends[ticker]['buy_opportunity'] = False

                # Sell opportunity:
                # 1. Stock has been going up but now turning down
                # 2. Stock hits take profit level
                # 3. Stock hits stop loss level
                # 4. Stock in prolonged downtrend
                if ticker in positions and positions[ticker]['shares'] > 0:
                    if (old_trend == 'up' and daily_change < -0.03) or \
                       (new_price >= positions[ticker]['take_profit']) or \
                       (new_price <= positions[ticker]['stop_loss']) or \
                       (old_trend == 'down' and ticker_trends[ticker]['days_in_trend'] >= 5):
                        ticker_trends[ticker]['sell_opportunity'] = True
                    else:
                        ticker_trends[ticker]['sell_opportunity'] = False

        # Generate buy/sell/hold recommendations based on trends
        actions_today = []

        # First, process sells to free up cash
        for ticker in list(positions.keys()):
            if ticker in ticker_trends and ticker_trends[ticker]['sell_opportunity'] and positions[ticker]['shares'] > 0:
                price = simulated_prices[ticker]
                shares = positions[ticker]['shares']
                avg_price = positions[ticker]['avg_price']
                cost_basis = positions[ticker]['cost_basis']

                if shares > 0:
                    # Calculate the actual proceeds after transaction fees
                    transaction_amount = shares * price

                    # Apply transaction fee and other costs
                    transaction_fee = transaction_amount * 0.001  # 0.1% fee
                    tax_rate = 0.15  # 15% tax on gains

                    # Calculate pre-tax profit
                    pre_tax_profit = transaction_amount - cost_basis - transaction_fee

                    # Calculate tax (only on profits, not losses)
                    tax = max(0, pre_tax_profit * tax_rate)

                    # Calculate final profit
                    profit = pre_tax_profit - tax
                    profit_percentage = (profit / cost_basis) * 100

                    # Add to cash balance
                    remaining_cash += transaction_amount - transaction_fee - tax

                    # Record the sell recommendation
                    sell_reason = ""
                    if price >= positions[ticker]['take_profit']:
                        sell_reason = f"Take profit target reached (+{risk_config['take_profit_pct']*100}%)"
                    elif price <= positions[ticker]['stop_loss']:
                        sell_reason = f"Stop loss triggered (-{risk_config['stop_loss_pct']*100}%)"
                    elif ticker_trends[ticker]['trend'] == 'up' and ticker_trends[ticker]['price_change'] > 0.15:
                        sell_reason = "Taking profits after strong rally"
                    elif ticker_trends[ticker]['trend'] == 'down' and ticker_trends[ticker]['days_in_trend'] >= 5:
                        sell_reason = "Cutting losses in extended downtrend"
                    else:
                        sell_reason = "Technical sell signal"

                    date_based_recommendations['sells'].append({
                        'stock': ticker,
                        'date': sim_date.strftime('%Y-%m-%d'),
                        'action': 'sell',
                        'shares': shares,
                        'price': price,
                        'transaction_fee': transaction_fee,
                        'tax': tax,
                        'gross_proceeds': transaction_amount,
                        'net_proceeds': transaction_amount - transaction_fee - tax,
                        'cost_basis': cost_basis,
                        'profit': profit,
                        'profit_percentage': profit_percentage,
                        'reason': sell_reason
                    })

                    # Update cumulative profit
                    cumulative_profit += profit

                    # Mark position as closed
                    positions[ticker]['shares'] = 0

                    # Update last action day
                    ticker_trends[ticker]['last_action_day'] = ticker_trends[ticker]['days_in_trend']

                    actions_today.append(f"SELL {ticker}: {shares:.2f} shares @ ${price:.2f} - Profit: ${profit:.2f} ({profit_percentage:.2f}%) - {sell_reason}")

        # Now, process buys with updated cash
        for ticker in ticker_trends:
            # Buy logic
            if ticker_trends[ticker]['buy_opportunity'] and (ticker not in positions or positions[ticker]['shares'] == 0):
                # Calculate how many shares we can buy
                price = simulated_prices[ticker]

                # Realistic position sizing based on risk profile:
                # 1. Never use more than max_position_pct% of portfolio on one position
                # 2. Ensure we have enough cash with reserve
                max_position_size = min(amount_cad * max_position_pct, remaining_cash * (1 - cash_reserve_pct))

                # Calculate number of shares (always round down to avoid fractional shares for some brokers)
                shares_to_buy = int(max_position_size / price)

                if shares_to_buy > 0:
                    # Calculate the actual cost including transaction fees
                    transaction_amount = shares_to_buy * price
                    transaction_fee = transaction_amount * 0.001  # 0.1% fee
                    total_cost = transaction_amount + transaction_fee

                    # Make sure we have enough cash
                    if total_cost <= remaining_cash:
                        # Deduct from cash
                        remaining_cash -= total_cost

                        # Record the buy recommendation
                        buy_reason = ""
                        if ticker_trends[ticker]['trend'] == 'down' and ticker_trends[ticker]['price_change'] < -0.1:
                            buy_reason = "Potential bottom (down trend reversal)"
                        elif ticker_trends[ticker]['trend'] == 'up' and ticker_trends[ticker]['days_in_trend'] >= 3:
                            buy_reason = "Strong uptrend continuation"
                        else:
                            buy_reason = "Technical buy signal"

                        date_based_recommendations['buys'].append({
                            'stock': ticker,
                            'date': sim_date.strftime('%Y-%m-%d'),
                            'action': 'buy',
                            'shares': shares_to_buy,
                            'price': price,
                            'transaction_fee': transaction_fee,
                            'total_cost': total_cost,
                            'profit': 0.0,
                            'profit_percentage': 0.0,
                            'reason': buy_reason
                        })

                        # Record the position with risk-adjusted stop loss and take profit
                        positions[ticker] = {
                            'shares': shares_to_buy,
                            'avg_price': price,
                            'cost_basis': total_cost,
                            'transaction_fee': transaction_fee,
                            'buy_date': sim_date,
                            'stop_loss': price * stop_loss_factor,
                            'take_profit': price * take_profit_factor,
                        }

                        # Update last action day
                        ticker_trends[ticker]['last_action_day'] = ticker_trends[ticker]['days_in_trend']

                        actions_today.append(f"BUY {ticker}: {shares_to_buy:.0f} shares @ ${price:.2f} - Cost: ${total_cost:.2f} - {buy_reason}")

            # Hold logic - record significant unrealized profits or losses
            elif ticker in positions and positions[ticker]['shares'] > 0:
                price = simulated_prices[ticker]
                shares = positions[ticker]['shares']
                avg_price = positions[ticker]['avg_price']
                cost_basis = positions[ticker]['cost_basis']
                days_held = (sim_date - positions[ticker]['buy_date']).days

                # Only record holds every 5 days or on significant price movements
                if days_held > 0 and (days_held % 5 == 0 or abs((price/avg_price - 1) * 100) > 10):
                    # Calculate unrealized profit (including estimated fees if sold now)
                    transaction_amount = shares * price
                    transaction_fee = transaction_amount * 0.001  # 0.1% fee
                    pre_tax_profit = transaction_amount - cost_basis - transaction_fee
                    tax = max(0, pre_tax_profit * 0.15)  # 15% tax on gains

                    unrealized_profit = pre_tax_profit - tax
                    unrealized_pct = (unrealized_profit / cost_basis) * 100

                    hold_reason = ""
                    if unrealized_pct > 15:
                        hold_reason = "Holding winning position, approaching target"
                    elif unrealized_pct < -10:
                        hold_reason = "Holding through drawdown, monitoring for reversal"
                    else:
                        hold_reason = "Maintaining position, no exit signal"

                    date_based_recommendations['holds'].append({
                        'stock': ticker,
                        'date': sim_date.strftime('%Y-%m-%d'),
                        'action': 'hold',
                        'shares': shares,
                        'price': price,
                        'days_held': days_held,
                        'cost_basis': cost_basis,
                        'current_value': transaction_amount,
                        'unrealized_profit': unrealized_profit,
                        'unrealized_percentage': unrealized_pct,
                        'reason': hold_reason
                    })

                    actions_today.append(f"HOLD {ticker}: {shares:.0f} shares, ${unrealized_profit:.2f} unrealized ({unrealized_pct:.2f}%) - {hold_reason}")

        # Print daily summary if there were actions
        if actions_today:
            print(f"\nDay {day} - {sim_date.strftime('%Y-%m-%d')} - Cash: ${remaining_cash:.2f}")
            for action in actions_today:
                print(f"  {action}")

    # Calculate remaining portfolio value
    final_portfolio_value = remaining_cash
    for ticker, position in positions.items():
        if position['shares'] > 0:
            final_portfolio_value += position['shares'] * simulated_prices[ticker]

    portfolio_gain = final_portfolio_value - amount_cad
    portfolio_gain_pct = (portfolio_gain / amount_cad) * 100

    print(f"\nSimulation complete.")
    print(f"Initial investment: ${amount_cad:.2f}")
    print(f"Final portfolio value: ${final_portfolio_value:.2f}")
    print(f"Total portfolio gain: ${portfolio_gain:.2f} ({portfolio_gain_pct:.2f}%)")
    print(f"Realized profits: ${cumulative_profit:.2f}")
    print(f"Remaining cash: ${remaining_cash:.2f}")
    print(f"Total buy recommendations: {len(date_based_recommendations['buys'])}")
    print(f"Total sell recommendations: {len(date_based_recommendations['sells'])}")
    print(f"Total hold notifications: {len(date_based_recommendations['holds'])}")

    return date_based_recommendations, final_portfolio_value, cumulative_profit

# Function to run the simulation with risk profile
def run_portfolio_simulation(recommender, amount, risk_profile='medium', days=30):
    """
    Run a portfolio simulation with specific risk profile

    Args:
        recommender: Portfolio recommender instance
        amount: Investment amount in CAD
        risk_profile: Risk profile ('low', 'medium', 'high')
        days: Number of days to simulate

    Returns:
        Simulation results
    """
    print(f"Running portfolio simulation with ${amount} investment and {risk_profile} risk profile")

    # Run the simulation with realistic profit calculations and risk management
    date_recommendations, final_value, realized_profit = recommender.simulate_future_recommendations_with_realistic_profits(
        amount_cad=amount,
        future_days=days,
        price_change_range=(-0.15, 0.25),
        risk_profile=risk_profile
    )

    # Display the chronological recommendations
    recommender.display_date_based_recommendations(date_recommendations)

    # Export to CSV
    recommender.export_date_based_recommendations(date_recommendations)

    # Create visualizations
    recommender.plot_trading_summary(date_recommendations)

    # Calculate additional performance metrics
    sells = date_recommendations['sells']
    if sells:
        # Calculate win rate
        winning_trades = sum(1 for sell in sells if sell['profit'] > 0)
        total_trades = len(sells)
        win_rate = (winning_trades / total_trades) * 100 if total_trades > 0 else 0

        # Calculate profit factor
        gross_profit = sum(sell['profit'] for sell in sells if sell['profit'] > 0)
        gross_loss = abs(sum(sell['profit'] for sell in sells if sell['profit'] <= 0))
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')

        # Calculate average holding period
        buys = {buy['stock']: buy['date'] for buy in date_recommendations['buys']}
        holding_periods = []

        for sell in sells:
            ticker = sell['stock']
            if ticker in buys:
                sell_date = datetime.strptime(sell['date'], '%Y-%m-%d')
                buy_date = datetime.strptime(buys[ticker], '%Y-%m-%d')
                hold_days = (sell_date - buy_date).days
                holding_periods.append(hold_days)

        avg_hold_days = sum(holding_periods) / len(holding_periods) if holding_periods else 0

        # Print the results
        print("\n=== ADVANCED PERFORMANCE METRICS ===")
        print(f"Risk Profile: {risk_profile}")
        print(f"Win Rate: {win_rate:.2f}%")
        print(f"Profit Factor: {profit_factor:.2f}")
        print(f"Average Holding Period: {avg_hold_days:.1f} days")
        print(f"ROI: {realized_profit/amount*100:.2f}%")
        print(f"Final Portfolio Value: ${final_value:.2f}")
        print(f"Portfolio Growth: {(final_value/amount - 1)*100:.2f}%")

    return date_recommendations, final_value, realized_profit

In [5]:
# Import necessary libraries
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from stable_baselines3 import DDPG
import os
import random

# Set paths to your model and data
model_path = "/content/drive/MyDrive/ddpg_portfolio_model.zip"
data_path = "/content/drive/MyDrive/historical_data.csv"

def create_portfolio_simulator_ui(recommender):
    """Create an enhanced UI for portfolio simulation with styling"""

    # Add CSS styling
    display(HTML("""
    <style>
        .widget-label {
            min-width: 20em !important;
            max-width: 20em !important;
        }
        .widget-inline-hbox {
            width: 90% !important;
        }
        .custom-title {
            font-family: 'Arial', sans-serif;
            color: #2c3e50;
            padding: 10px 0;
            margin-bottom: 20px;
            border-bottom: 2px solid #3498db;
        }
        .section-title {
            font-family: 'Arial', sans-serif;
            color: #2c3e50;
            margin: 15px 0 5px 0;
            font-weight: bold;
        }
        .ui-wrapper {
            background-color: #f8f9fa;
            border-radius: 8px;
            padding: 20px;
            margin-bottom: 20px;
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1);
        }
        .risk-description {
            margin-top: 10px;
            padding: 10px;
            background-color: #e8f4f8;
            border-radius: 5px;
            border-left: 4px solid #3498db;
        }
    </style>
    """))

    # Display title
    display(HTML("<h1 class='custom-title'>Portfolio Simulation Tool</h1>"))
    display(HTML("<div class='ui-wrapper'>"))
    display(HTML("<div class='section-title'>Investment Parameters</div>"))

    # Create widgets for user input
    investment_amount_widget = widgets.IntSlider(
        value=10000,
        min=1000,
        max=100000,
        step=1000,
        description='Investment Amount ($):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='90%')
    )

    investment_amount_text = widgets.IntText(
        value=10000,
        layout=widgets.Layout(width='150px')
    )

    widgets.link((investment_amount_widget, 'value'), (investment_amount_text, 'value'))
    investment_box = widgets.HBox([investment_amount_widget, investment_amount_text])

    days_widget = widgets.IntSlider(
        value=30,
        min=7,
        max=90,
        step=1,
        description='Simulation Days:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='90%')
    )

    days_text = widgets.IntText(
        value=30,
        layout=widgets.Layout(width='150px')
    )

    widgets.link((days_widget, 'value'), (days_text, 'value'))
    days_box = widgets.HBox([days_widget, days_text])

    display(investment_box)
    display(days_box)

    display(HTML("<div class='section-title'>Risk Profile</div>"))

    # Create risk profile widget with descriptions
    risk_profile_widget = widgets.RadioButtons(
        options=[
            ('Conservative (Low Risk)', 'low'),
            ('Moderate (Medium Risk)', 'medium'),
            ('Aggressive (High Risk)', 'high')
        ],
        value='medium',
        description='Risk Profile:',
        style={'description_width': 'initial'}
    )

    # Risk profile descriptions
    risk_descriptions = {
        'low': """
            <div class='risk-description'>
                <strong>Conservative Strategy:</strong><br>
                • Stop Loss: 10%<br>
                • Take Profit: 15%<br>
                • Max Position Size: 3% of portfolio<br>
                • Cash Reserve: 30% of available cash<br>
                <em>Best for capital preservation with modest growth expectations.</em>
            </div>
        """,
        'medium': """
            <div class='risk-description'>
                <strong>Moderate Strategy:</strong><br>
                • Stop Loss: 15%<br>
                • Take Profit: 20%<br>
                • Max Position Size: 5% of portfolio<br>
                • Cash Reserve: 25% of available cash<br>
                <em>Balanced approach for reasonable growth with moderate risk tolerance.</em>
            </div>
        """,
        'high': """
            <div class='risk-description'>
                <strong>Aggressive Strategy:</strong><br>
                • Stop Loss: 20%<br>
                • Take Profit: 30%<br>
                • Max Position Size: 8% of portfolio<br>
                • Cash Reserve: 15% of available cash<br>
                <em>Higher risk strategy seeking maximum growth potential.</em>
            </div>
        """
    }

    risk_description_widget = widgets.HTML(value=risk_descriptions['medium'])

    def update_risk_description(change):
        risk_description_widget.value = risk_descriptions[change['new']]

    risk_profile_widget.observe(update_risk_description, names='value')

    display(risk_profile_widget)
    display(risk_description_widget)

    # Create output area
    output = widgets.Output()

    def run_simulation_button_clicked(b):
        with output:
            clear_output()
            print(f"Running portfolio simulation with ${investment_amount_widget.value:,} investment")
            print(f"Duration: {days_widget.value} days")
            print(f"Risk profile: {risk_profile_widget.value}")
            print("-" * 50)

            # Run simulation
            run_portfolio_simulation(
                recommender,
                investment_amount_widget.value,
                risk_profile_widget.value,
                days_widget.value
            )

    # Create a button widget
    run_button = widgets.Button(
        description='Run Simulation',
        button_style='success',
        tooltip='Click to run the portfolio simulation',
        icon='play',
        layout=widgets.Layout(width='200px', height='40px')
    )

    # Link button click to function
    run_button.on_click(run_simulation_button_clicked)

    # Display button with padding
    display(HTML("<div style='padding: 20px 0;'>"))
    display(run_button)
    display(HTML("</div>"))
    display(HTML("</div>"))  # Close the ui-wrapper div

    # Output area with styling
    display(HTML("<div class='ui-wrapper'>"))
    display(HTML("<div class='section-title'>Simulation Results</div>"))
    display(output)
    display(HTML("</div>"))

# Function to set up and run the UI
def setup_portfolio_simulator():
    # Create the recommender instance
    recommender = DDPGPortfolioRecommender(
        model_path=model_path,
        data_path=data_path,
        max_stocks=100,
        lookback=30,
        feature_dimension=7
    )

    # Add the methods to the class
    DDPGPortfolioRecommender.simulate_future_recommendations_with_realistic_profits = simulate_future_recommendations_with_realistic_profits
    DDPGPortfolioRecommender.display_date_based_recommendations = display_date_based_recommendations
    DDPGPortfolioRecommender.export_date_based_recommendations = export_date_based_recommendations
    DDPGPortfolioRecommender.plot_trading_summary = plot_trading_summary

    # Create and display the UI
    create_portfolio_simulator_ui(recommender)

# Call the function to display the UI
if __name__ == "__main__":
    # First, mount Google Drive if needed
    from google.colab import drive
    drive.mount('/content/drive')

    # Set up and run the portfolio simulator UI
    setup_portfolio_simulator()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading model from: /content/drive/MyDrive/ddpg_portfolio_model.zip


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 168.41GB > 11.60GB
  warnings.warn(


Loading data from: /content/drive/MyDrive/historical_data.csv
Latest data date: 2024-12-30 00:00:00
Number of tickers selected: 100
Preparing features...
Calculating features for 100 stocks with 7 features...
Feature calculation complete!
Initialization complete!


RadioButtons(description='Risk Profile:', index=1, options=(('Conservative (Low Risk)', 'low'), ('Moderate (Me…

HTML(value="\n            <div class='risk-description'>\n                <strong>Moderate Strategy:</strong><…

Button(button_style='success', description='Run Simulation', icon='play', layout=Layout(height='40px', width='…

Output()

In [6]:
# Import necessary libraries
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from stable_baselines3 import DDPG
import os
import random

# Set paths to your model and data
model_path = "/content/drive/MyDrive/ddpg_portfolio_model.zip"
data_path = "/content/drive/MyDrive/historical_data.csv"

def create_portfolio_simulator_ui(recommender):
    """Create a UI for portfolio simulation with user inputs for investment amount, days and risk profile"""

    # Create widgets for user input
    investment_amount_widget = widgets.IntSlider(
        value=10000,
        min=1000,
        max=100000,
        step=1000,
        description='Investment Amount ($):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )

    days_widget = widgets.IntSlider(
        value=30,
        min=7,
        max=90,
        step=1,
        description='Simulation Days:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )

    risk_profile_widget = widgets.RadioButtons(
        options=[
            ('Conservative (Low Risk)', 'low'),
            ('Moderate (Medium Risk)', 'medium'),
            ('Aggressive (High Risk)', 'high')
        ],
        value='medium',
        description='Risk Profile:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )

    output = widgets.Output()

    def run_simulation_button_clicked(b):
        with output:
            clear_output()
            print("Running simulation...")

            # Get values from widgets
            amount = investment_amount_widget.value
            days = days_widget.value
            risk_profile = risk_profile_widget.value

            # Run simulation
            run_portfolio_simulation(recommender, amount, risk_profile, days)

    # Create a button widget
    run_button = widgets.Button(
        description='Run Simulation',
        button_style='success',
        tooltip='Click to run the portfolio simulation',
        icon='play'
    )

    # Link button click to function
    run_button.on_click(run_simulation_button_clicked)

    # Display widgets
    display(widgets.HTML("<h2>Portfolio Simulation Settings</h2>"))
    display(investment_amount_widget)
    display(days_widget)
    display(risk_profile_widget)
    display(run_button)
    display(output)

# Function to set up and run the UI
def setup_portfolio_simulator():
    # Create the recommender instance
    recommender = DDPGPortfolioRecommender(
        model_path=model_path,
        data_path=data_path,
        max_stocks=100,
        lookback=30,
        feature_dimension=7
    )

    # Add the methods to the class
    DDPGPortfolioRecommender.simulate_future_recommendations_with_realistic_profits = simulate_future_recommendations_with_realistic_profits
    DDPGPortfolioRecommender.display_date_based_recommendations = display_date_based_recommendations
    DDPGPortfolioRecommender.export_date_based_recommendations = export_date_based_recommendations
    DDPGPortfolioRecommender.plot_trading_summary = plot_trading_summary

    # Create and display the UI
    create_portfolio_simulator_ui(recommender)

# Main execution
if __name__ == "__main__":
    # First, mount Google Drive if needed
    from google.colab import drive
    drive.mount('/content/drive')

    # Set up and run the portfolio simulator UI
    setup_portfolio_simulator()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading model from: /content/drive/MyDrive/ddpg_portfolio_model.zip


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 168.41GB > 11.09GB
  warnings.warn(


Loading data from: /content/drive/MyDrive/historical_data.csv
Latest data date: 2024-12-30 00:00:00
Number of tickers selected: 100
Preparing features...
Calculating features for 100 stocks with 7 features...
Feature calculation complete!
Initialization complete!


HTML(value='<h2>Portfolio Simulation Settings</h2>')

IntSlider(value=10000, description='Investment Amount ($):', layout=Layout(width='500px'), max=100000, min=100…

IntSlider(value=30, description='Simulation Days:', layout=Layout(width='500px'), max=90, min=7, style=SliderS…

RadioButtons(description='Risk Profile:', index=1, layout=Layout(width='500px'), options=(('Conservative (Low …

Button(button_style='success', description='Run Simulation', icon='play', style=ButtonStyle(), tooltip='Click …

Output()

In [8]:
# Import necessary libraries
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from stable_baselines3 import DDPG
import os
import random

# Set paths to your model and data
model_path = "/content/drive/MyDrive/ddpg_portfolio_model.zip"
data_path = "/content/drive/MyDrive/historical_data.csv"

# UI with input fields instead of sliders
def create_portfolio_simulator_ui(recommender):
    """Create a UI for portfolio simulation with user inputs for investment amount, days and risk profile"""

    # Create input fields instead of sliders
    investment_amount_widget = widgets.BoundedIntText(
        value=10000,
        min=1000,
        max=100000,
        step=1000,
        description='Investment Amount ($):',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )

    days_widget = widgets.BoundedIntText(
        value=30,
        min=7,
        max=90,
        step=1,
        description='Simulation Days:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )

    risk_profile_widget = widgets.RadioButtons(
        options=[
            ('Conservative (Low Risk)', 'low'),
            ('Moderate (Medium Risk)', 'medium'),
            ('Aggressive (High Risk)', 'high')
        ],
        value='medium',
        description='Risk Profile:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )

    output = widgets.Output()

    def run_simulation_button_clicked(b):
        with output:
            clear_output()
            print("Running simulation...")

            # Get values from widgets
            amount = investment_amount_widget.value
            days = days_widget.value
            risk_profile = risk_profile_widget.value

            # Run simulation
            run_portfolio_simulation(recommender, amount, risk_profile, days)

    # Create a button widget
    run_button = widgets.Button(
        description='Run Simulation',
        button_style='success',
        tooltip='Click to run the portfolio simulation',
        icon='play'
    )

    run_button.on_click(run_simulation_button_clicked)

    # Organize layout in a vertical stack
    input_form = widgets.VBox([
        investment_amount_widget,
        days_widget,
        risk_profile_widget,
        run_button
    ])

    # Display UI
    display(widgets.HTML("<h2>Portfolio Simulation Settings</h2>"))
    display(input_form)
    display(output)

# Function to set up and run the UI
def setup_portfolio_simulator():
    # Create the recommender instance
    recommender = DDPGPortfolioRecommender(
        model_path=model_path,
        data_path=data_path,
        max_stocks=100,
        lookback=30,
        feature_dimension=7
    )

    # Attach custom methods (make sure these are defined elsewhere in your notebook/script)
    DDPGPortfolioRecommender.simulate_future_recommendations_with_realistic_profits = simulate_future_recommendations_with_realistic_profits
    DDPGPortfolioRecommender.display_date_based_recommendations = display_date_based_recommendations
    DDPGPortfolioRecommender.export_date_based_recommendations = export_date_based_recommendations
    DDPGPortfolioRecommender.plot_trading_summary = plot_trading_summary

    # Create and display the UI
    create_portfolio_simulator_ui(recommender)
setup_portfolio_simulator()

Loading model from: /content/drive/MyDrive/ddpg_portfolio_model.zip


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/buffers.py:242: UserWarning: This system does not have apparently enough memory to store the complete replay buffer 168.41GB > 10.56GB
  warnings.warn(


Loading data from: /content/drive/MyDrive/historical_data.csv
Latest data date: 2024-12-30 00:00:00
Number of tickers selected: 100
Preparing features...
Calculating features for 100 stocks with 7 features...
Feature calculation complete!
Initialization complete!


HTML(value='<h2>Portfolio Simulation Settings</h2>')

Output()